`conda activate multi_integration`

# Joint analysis of paired and unpaired multiomic data with scGLUE

## Data preprocessing

In [1]:
import anndata as ad
import networkx as nx
import scanpy as sc
import pandas as pd
import numpy as np
import muon as mu
import scglue
import os
from matplotlib import rcParams
import matplotlib.pyplot as plt

import seaborn as sns
from itertools import chain
from random import Random

# Import a module with ATAC-seq-related functions
from muon import atac as ac

/home/cruiz2/miniconda3/envs/multi_integration/lib/python3.9/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/cruiz2/miniconda3/envs/multi_integration/lib/python3.9/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/cruiz2/miniconda3/envs/multi_integration/l

In [2]:
sc.settings.n_jobs = 32

In [3]:
scglue.plot.set_publication_params()
rcParams["figure.figsize"] = (4, 4)

In [4]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available")
    
    # Get the number of available GPUs
    num_gpus = torch.cuda.device_count()
    print(f"Number of available GPUs: {num_gpus}")
    
    # Get the name of each available GPU
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        print(f"GPU {i}: {gpu_name}")
else:
    print("CUDA is not available")

CUDA is not available


### Preprocess scRNA-seq data

In [5]:
rna = sc.read('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered_and_normalized.h5ad')
rna.obs['domain'] = 'scRNA-seq'
rna

AnnData object with n_obs × n_vars = 409561 × 19248
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_RAW', 'nFeature_RAW', 'DF.class', 'DF.score', 'scDblFinder.class', 'scDblFinder.score', 'ID', 'SampleID', 'Data', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'nFeature_Diff', 'nCount_Diff', 'Batch_for_correction', 'scDblFinder.clusters.class', 'scDblFinder.clusters.score', 'scDblFinder.class.clusters', 'doublet.combn.fisher', 'doublet.combn.mean', 'doublet.combn.fisher.class', 'doublet.combn.mean.class', 'Study', 'Original_annotation', 'Isolation_method_by_cell', 'Cell_type_granular_mouse_correlations', 'Cell_type_mouse_correlations', 'Cell_type_consensus_Jessa2022', 'Malignant_normal_consensus_Jessa2022', 'Institute', 'Preservation_method', 'Diagnosis', 'Tumor_type', 'Tumor_subtype', 'Location', 'Source', 'Clinical_status', 'Isolation_method', 'Sc_platform_RNA', 'Sc_platform_ATAC', 'Sc_multiome', 'Raw_data_available', 'Counts', 'Genome_version', 'Paired_sampl

In [9]:
rna.X, rna.X.data

(<397794x19248 sparse matrix of type '<class 'numpy.float32'>'
 	with 835603632 stored elements in Compressed Sparse Column format>,
 array([5., 1., 2., ..., 1., 1., 1.], dtype=float32))

In [6]:
rna.X = rna.layers['scran_normalization'].copy()
rna.X, rna.X.data

(<409561x19248 sparse matrix of type '<class 'numpy.float64'>'
 	with 891374044 stored elements in Compressed Sparse Row format>,
 array([0.37975201, 0.86952024, 0.86952024, ..., 3.25687059, 3.25687059,
        3.25687059]))

In [7]:
del rna.layers

In [9]:
rna.write("/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered_and_scran_normalized_for_sceasy_conversion.h5ad", compression="gzip")